In [12]:
import json
import xml.etree.ElementTree as ET
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

IMAGE_DIR = r'frames-batch-5/frames'
JSON_FILE = r'frames-batch-5/mediapipe_landmarks.json'
XML_FILE = r'frames-batch-5/annotations.xml'
OUTPUT_JSON = r'frames-batch-5/pixel_errors_by_joint.json'
OUTPUT_FOLDER = r'frames-batch-5/skeletons_comparison'

MP_JOINTS = [
    "WRIST", "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_FINGER_MCP", "INDEX_FINGER_PIP", "INDEX_FINGER_DIP", "INDEX_FINGER_TIP",
    "MIDDLE_FINGER_MCP", "MIDDLE_FINGER_PIP", "MIDDLE_FINGER_DIP", "MIDDLE_FINGER_TIP",
    "RING_FINGER_MCP", "RING_FINGER_PIP", "RING_FINGER_DIP", "RING_FINGER_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP"
]

JOINT_TO_IDX = {name: i for i, name in enumerate(MP_JOINTS)}

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),           # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),           # Index
    (0, 9), (9, 10), (10, 11), (11, 12),      # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),    # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)     # Pinky
]

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def draw_skeleton(ax, joints, connections, color='green', marker='o', label=None):
    xs, ys = [], []
    for pt in joints.values():
        xs.append(pt[0])
        ys.append(pt[1])
    ax.scatter(xs, ys, c=color, marker=marker, label=label, s=30, edgecolors='black')

    for s, e in connections:
        start_name, end_name = MP_JOINTS[s], MP_JOINTS[e]
        if start_name in joints and end_name in joints:
            x_coords = [joints[start_name][0], joints[end_name][0]]
            y_coords = [joints[start_name][1], joints[end_name][1]]
            ax.plot(x_coords, y_coords, c=color, linewidth=2)

def run_comparison():
    with open(JSON_FILE, 'r') as f:
        mp_data = json.load(f)
    tree = ET.parse(XML_FILE)
    root = tree.getroot()

    xml_frames = set(os.path.basename(img.get('name')) for img in root.findall('image'))
    json_frames = set(mp_data.keys())
    common_frames = xml_frames & json_frames

    all_errors = {}

    for image_tag in root.findall('image'):
        frame_name = os.path.basename(image_tag.get('name'))
        if frame_name not in common_frames:
            continue

        width, height = int(image_tag.get('width')), int(image_tag.get('height'))
        img_path = os.path.join(IMAGE_DIR, frame_name)
        canvas = cv2.imread(img_path)
        if canvas is None:
            canvas = np.zeros((height, width, 3), dtype=np.uint8)

        frame_errors = {}
        fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)
        ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        ax.axis('off')

        for skeleton in image_tag.findall('skeleton'):
            hand_type = skeleton.get('label')  # 'left_hand' / 'right_hand'
            if hand_type not in mp_data[frame_name]:
                continue

            mp_hand_raw = mp_data[frame_name][hand_type]
            gt_pixels, mp_pixels, hand_errs = {}, {}, {}

            for p_tag in skeleton.findall('points'):
                label = p_tag.get('label')
                idx = JOINT_TO_IDX.get(label)
                if idx is None or idx >= len(mp_hand_raw):
                    continue

                gx, gy = map(float, p_tag.get('points').split(','))
                mx, my = mp_hand_raw[idx]['x'] * width, mp_hand_raw[idx]['y'] * height

                gt_pixels[label] = (gx, gy)
                mp_pixels[label] = (mx, my)
                hand_errs[label] = round(float(np.linalg.norm([gx - mx, gy - my])), 2)

            if gt_pixels and mp_pixels:
                if hand_type == 'left_hand':
                    draw_skeleton(ax, gt_pixels, HAND_CONNECTIONS, color='green', marker='o', label='GT Left')
                    draw_skeleton(ax, mp_pixels, HAND_CONNECTIONS, color='blue', marker='x', label='MP Left')
                else:
                    draw_skeleton(ax, gt_pixels, HAND_CONNECTIONS, color='cyan', marker='o', label='GT Right')
                    draw_skeleton(ax, mp_pixels, HAND_CONNECTIONS, color='purple', marker='x', label='MP Right')

                frame_errors[hand_type] = hand_errs

        # Legend
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc='upper right')

        if frame_errors:
            all_errors[frame_name] = frame_errors
            plt.tight_layout()
            output_path = os.path.join(OUTPUT_FOLDER, f"skel_{frame_name}")
            plt.savefig(output_path)
        plt.close(fig)

    with open(OUTPUT_JSON, 'w') as f:
        json.dump(all_errors, f, indent=4)
    print(f"Done! Images saved in '{OUTPUT_FOLDER}', JSON saved as '{OUTPUT_JSON}'")

if __name__ == "__main__":
    run_comparison()

Done! Images saved in 'frames-batch-5/skeletons_comparison', JSON saved as 'frames-batch-5/pixel_errors_by_joint.json'
